<a href="https://colab.research.google.com/github/tranduchoa518-lang/fake-news-detection/blob/main/notebooks/05_welfake_dataset_loading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file == 'WELFake_Dataset.csv':
            print(os.path.join(root, file))

/content/drive/MyDrive/WELFake_Dataset.csv


In [3]:
import pandas as pd

file_path = '/content/drive/MyDrive/WELFake_Dataset.csv'

df = pd.read_csv(file_path)

print("Kích thước dữ liệu:", df.shape)
display(df.head())

Kích thước dữ liệu: (72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [4]:
print("Số dòng:", len(df))

print("\nSố lượng giá trị bị thiếu:")
print(df.isnull().sum())

print("\nPhân bố nhãn:")
print(df['label'].value_counts())

Số dòng: 72134

Số lượng giá trị bị thiếu:
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

Phân bố nhãn:
label
1    37106
0    35028
Name: count, dtype: int64


In [5]:

# ==========================================
# 1. LÀM SẠCH DỮ LIỆU WELFAKE
# ==========================================

import pandas as pd

# Tạo bản sao dữ liệu
df_clean = df.copy()

# Chuẩn hóa tên cột
df_clean.columns = df_clean.columns.str.strip().str.lower()

# Kiểm tra các cột cần thiết
required_columns = ['title', 'text', 'label']

for col in required_columns:
    if col not in df_clean.columns:
        raise ValueError(f"Không tìm thấy cột {col}")

# Xử lý dữ liệu thiếu
df_clean['title'] = df_clean['title'].fillna('').astype(str)
df_clean['text'] = df_clean['text'].fillna('').astype(str)

# Kết hợp tiêu đề và nội dung
df_clean['text'] = (
    df_clean['title'] + ' ' + df_clean['text']
).str.strip()

# Chuyển nhãn về dạng số
df_clean['label'] = pd.to_numeric(
    df_clean['label'], errors='coerce'
)

# Loại bỏ dữ liệu thiếu và nhãn không hợp lệ
df_clean = df_clean.dropna(subset=['label'])
df_clean['label'] = df_clean['label'].astype(int)

df_clean = df_clean[df_clean['label'].isin([0, 1])]

# Loại bỏ bài báo không có nội dung
df_clean = df_clean[df_clean['text'] != '']

# Xóa dữ liệu trùng lặp TRƯỚC KHI chia tập
df_clean = df_clean.drop_duplicates(
    subset=['text']
)

# Chỉ giữ hai cột cần thiết
df_clean = df_clean[['text', 'label']]

# Đặt lại chỉ mục
df_clean = df_clean.reset_index(drop=True)

# Kiểm tra kết quả
print("Dữ liệu sau khi làm sạch:", df_clean.shape)
print("\nPhân bố nhãn:")
print(df_clean['label'].value_counts())

print("\nSố dòng trùng còn lại:",
      df_clean.duplicated(subset=['text']).sum())

display(df_clean.head())


Dữ liệu sau khi làm sạch: (63676, 2)

Phân bố nhãn:
label
0    34790
1    28886
Name: count, dtype: int64

Số dòng trùng còn lại: 0


,text,label
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,1
1,Did they post their votes for Hillary already?,1
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,1
3,"Bobby Jindal, raised Hindu, uses story of Chri...",0
4,SATAN 2: Russia unvelis an image of its terrif...,1


In [6]:

# ==========================================
# 2. CHIA TRAIN - VALIDATION - TEST
# ==========================================

from sklearn.model_selection import train_test_split

# Chia 70% Train và 30% dữ liệu còn lại
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
    stratify=df_clean['label']
)

# Chia 30% còn lại thành Validation 15% và Test 15%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['label']
)

# Kiểm tra số lượng dữ liệu
total = len(df_clean)

print("Tổng dữ liệu:", total)

print("\nTrain:", len(train_df),
      f"({len(train_df)/total*100:.2f}%)")

print("Validation:", len(val_df),
      f"({len(val_df)/total*100:.2f}%)")

print("Test:", len(test_df),
      f"({len(test_df)/total*100:.2f}%)")

# Kiểm tra phân bố nhãn
print("\nPhân bố nhãn Train:")
print(train_df['label'].value_counts())

print("\nPhân bố nhãn Validation:")
print(val_df['label'].value_counts())

print("\nPhân bố nhãn Test:")
print(test_df['label'].value_counts())


Tổng dữ liệu: 63676

Train: 44573 (70.00%)
Validation: 9551 (15.00%)
Test: 9552 (15.00%)

Phân bố nhãn Train:
label
0    24353
1    20220
Name: count, dtype: int64

Phân bố nhãn Validation:
label
0    5218
1    4333
Name: count, dtype: int64

Phân bố nhãn Test:
label
0    5219
1    4333
Name: count, dtype: int64


In [7]:

# ==========================================
# 3. LƯU DỮ LIỆU ĐÃ XỬ LÝ
# ==========================================

import os

# Thư mục lưu dữ liệu thống nhất
OUTPUT_DIR = "/content/drive/MyDrive/fake-news-detection/data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lưu 3 tập dữ liệu
train_df.to_csv(
    os.path.join(OUTPUT_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(OUTPUT_DIR, "val.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(OUTPUT_DIR, "test.csv"),
    index=False
)

print("ĐÃ LƯU THÀNH CÔNG!")

print("\nCác tệp đã lưu:")
print(os.listdir(OUTPUT_DIR))


ĐÃ LƯU THÀNH CÔNG!

Các tệp đã lưu:
['train.csv', 'val.csv', 'test.csv', 'isot_combined.csv', 'isot_train.csv', 'isot_val.csv', 'isot_test.csv']


In [8]:

# ==========================================
# 4. KIỂM TRA DỮ LIỆU ĐÃ LƯU
# ==========================================

train_check = pd.read_csv(
    os.path.join(OUTPUT_DIR, "train.csv")
)

val_check = pd.read_csv(
    os.path.join(OUTPUT_DIR, "val.csv")
)

test_check = pd.read_csv(
    os.path.join(OUTPUT_DIR, "test.csv")
)

print("Train:", train_check.shape)
print("Validation:", val_check.shape)
print("Test:", test_check.shape)

print("\nCác cột dữ liệu:")
print(train_check.columns.tolist())

print("\n5 dòng Train:")
display(train_check.head())

print("\n5 dòng Validation:")
display(val_check.head())

print("\n5 dòng Test:")
display(test_check.head())


Train: (44573, 2)
Validation: (9551, 2)
Test: (9552, 2)

Các cột dữ liệu:
['text', 'label']

5 dòng Train:


,text,label
0,Putin vows to 'choke' NATO [Video] November 20...,1
1,Secretive GOP Group Targets Trump for Destruct...,0
2,LONDON’S FIRST MUSLIM MAYOR BANS “Sexy Women” ...,1
3,A One-Cent Soda Tax Gets Expensive in Californ...,0
4,Trump Might Have Just Put Himself In Prison Wi...,1



5 dòng Validation:


,text,label
0,TRUMP SUPPORTER Assaulted Outside Rally In San...,1
1,Trump Attacks Senator’s Credibility Over Gorsu...,0
2,Ted Cruz Has A Sickening New Plan For America’...,1
3,History class becomes a debate on America The ...,0
4,“Dead Broke” Clintons Hanging At 21 Club In NY...,1



5 dòng Test:


,text,label
0,Facebook to Stop ’Fake News’ Same Way as ’Clic...,0
1,Protests Erupt In Charlotte After Cops Kill Di...,1
2,President Trump: Man-Made Global Warming is a ...,1
3,2:00PM Water Cooler 11/7/2016 D 20 \nLegend: L...,1
4,Incredible smoke haze seen outside NDTV office...,1
